# 5. Multi Head Attention

**Senaryo (Örnek) : **

Multi-Head Attention: Aynı cümleyi okuyan bir uzmanlar komitesi gibidir.

Kafa 1 (Dilbilgisi Uzmanı): Sadece cümlenin dilbilgisi yapısına, özne-yüklem uyumuna bakar.

Kafa 2 (Anlam Uzmanı): Kelimelerin anlamsal yakınlığına, eş ve zıt anlamlılarına odaklanır.

Kafa 3 (Bağlam Uzmanı): Zamirlerin kimi işaret ettiğine ("o", "bunu" vb.) veya cümlenin genel akışına bakar.

Her uzman (kafa), aynı metne bakar ancak kendi uzmanlık filtresinden (kendi W_q, W_k, W_v seti) geçirerek analiz eder. Sonunda, tüm uzmanların raporları birleştirilerek cümlenin çok daha zengin ve çok yönlü bir analizi ortaya çıkar.

**Farklılaşma Nerede Oluyor?**

Her kafanın Q, K, V matrislerinin farklı olmasının sebebi, her kafa için ayrı ve bağımsız Ağırlık Matrisleri (W_q, W_k, W_v) öğrenilmesidir. Farklılaşma tam olarak bu noktada başlar. Model, her kafanın farklı bir ilişki türünü öğrenmesi için onu farklı ağırlık setleriyle donatır.

In [4]:
import numpy as np

# --- Adım 0: Temel Ayarlar ve Hiperparametreler ---
print("--- Adım 0: Ayarlar ve Hiperparametreler ---")
ornek_metin = "akıllı köpek kırmızı topu kovaladı"
tokenler = ornek_metin.split()
print(f"Girdi Metni: '{ornek_metin}'\n")

# Modelin genel vektör boyutu
d_model = 6
# Kullanılacak kafa sayısı
num_heads = 6

# d_model, kafa sayısına tam bölünmeli. Bu bir mimari kuralıdır.
# Her kafa daha küçük bir boyutta çalışır.
d_k = d_model // num_heads

print(f"Model Boyutu (d_model): {d_model}")
print(f"Kafa Sayısı (num_heads): {num_heads}")
print(f"Her Kafanın Boyutu (d_k): {d_model} / {num_heads} = {d_k}\n")

# Tekrarlanabilirlik için
np.random.seed(42)

# Giriş kelimeleri için embedding'ler (tek ve ortak)
X = np.round(np.random.rand(len(tokenler), d_model), 2)
print(f"Ortak Giriş Embedding Matrisi X (boyut: {X.shape}):\n{X}\n")
print("="*60)

# --- Adım 1: Her Kafa İçin Ayrı Ağırlık Matrisleri Oluşturma ---
print("\n--- Adım 1: Her Kafa İçin AYRI Ağırlık Matrisleri Oluşturma ---")
print("Farklılaşmanın başladığı yer tam olarak burasıdır!")
print(f"{num_heads} kafa olduğu için, {num_heads} set ayrı W_q, W_k, W_v matrisi oluşturulur.\n")

kafa_agirliklari = {}
for i in range(num_heads):
    kafa_agirliklari[f'kafa_{i+1}'] = {
        'W_q': np.round(np.random.rand(d_model, d_k), 2),
        'W_k': np.round(np.random.rand(d_model, d_k), 2),
        'W_v': np.round(np.random.rand(d_model, d_k), 2)
    }

# Oluşturulan ağırlıkları gösterelim
for i in range(num_heads):
    print(f"--- BAŞLIK {i+1} AĞIRLIKLARI ---")
    print(f"W_q_{i+1} (boyut: {kafa_agirliklari[f'kafa_{i+1}']['W_q'].shape}):\n{kafa_agirliklari[f'kafa_{i+1}']['W_q']}\n")
    print(f"W_k_{i+1} (boyut: {kafa_agirliklari[f'kafa_{i+1}']['W_k'].shape}):\n{kafa_agirliklari[f'kafa_{i+1}']['W_k']}\n")
    
print("Gördüğünüz gibi, her kafanın kendine özgü, rastgele başlatılmış ağırlıkları var.")
print("Eğitim sırasında bu ağırlıklar farklılaşarak farklı görevlerde uzmanlaşır.")
print("="*60)


# --- Adım 2: Her Kafanın Paralel Olarak Kendi Q,K,V'sini Hesaplaması ---
print("\n--- Adım 2: Her Kafa, AYNI GİRDİYİ Kendi Ağırlıklarıyla İşliyor ---")

kafa_ciktilari = {}

for i in range(num_heads):
    kafa_adi = f'kafa_{i+1}'
    print(f"\n--- BAŞLIK {i+1} İŞLEMDE ---")
    
    # İlgili kafanın ağırlıklarını al
    W_q = kafa_agirliklari[kafa_adi]['W_q']
    W_k = kafa_agirliklari[kafa_adi]['W_k']
    W_v = kafa_agirliklari[kafa_adi]['W_v']
    
    # ORTAK GİRDİ (X) ile bu kafaya ÖZEL ağırlıkları çarparak Q, K, V'yi oluştur
    print(f"Ortak X matrisi, BAŞLIK {i+1}'e özel W_q, W_k, W_v ile çarpılıyor...")
    Q = np.dot(X, W_q)
    K = np.dot(X, W_k)
    V = np.dot(X, W_v)
    
    print(f"\nBAŞLIK {i+1} için üretilen Q Matrisi (boyut: {Q.shape}):\n{np.round(Q, 2)}")
    print(f"BAŞLIK {i+1} için üretilen K Matrisi (boyut: {K.shape}):\n{np.round(K, 2)}")
    print(f"BAŞLIK {i+1} için üretilen V Matrisi (boyut: {V.shape}):\n{np.round(V, 2)}")
    
    # Bu kafanın dikkat skorlarını ve çıktısını hesapla
    skorlar = np.dot(Q, K.T) / np.sqrt(d_k)
    dikkat_agirliklari = np.exp(skorlar) / np.sum(np.exp(skorlar), axis=-1, keepdims=True)
    
    # Her kafanın kendi dikkat çıktısı var
    kafa_ciktisi = np.dot(dikkat_agirliklari, V)
    kafa_ciktilari[kafa_adi] = kafa_ciktisi
    
    print(f"\nBAŞLIK {i+1}'in kendi dikkat çıktısı (boyut: {kafa_ciktisi.shape}):\n{np.round(kafa_ciktisi, 2)}")
    
print("\nFarklı ağırlıklar nedeniyle her kafanın ürettiği Q, K, V matrisleri ve sonuçta ürettiği dikkat çıktısı birbirinden farklıdır.")
print("="*60)

# --- Adım 3: Kafalardan Gelen Çıktıları Birleştirme ve Final Çıktı ---
print("\n--- Adım 3: Kafaların Raporlarını (Çıktılarını) Birleştirme ---")

# Her kafadan gelen (5, 3) boyutundaki çıktıları yanyana birleştir (concatenate)
# Sonuçta (5, 6) yani (token_sayisi, d_model) boyutunda bir matris elde ederiz.
birlesik_cikti = np.concatenate(list(kafa_ciktilari.values()), axis=1)
print(f"Tüm kafa çıktılarının birleştirilmiş hali (boyut: {birlesik_cikti.shape}):\n{np.round(birlesik_cikti, 2)}\n")

# Son bir dokunuş: Bu birleşik çıktı, genellikle son bir ağırlık matrisi (W_o) ile daha çarpılır.
# Bu matris, farklı kafalardan gelen bilgileri harmanlamayı öğrenir.
W_o = np.round(np.random.rand(d_model, d_model), 2)
print(f"Tüm bilgileri harmanlayan final Ağırlık Matrisi W_o (boyut: {W_o.shape}):\n{W_o}\n")

final_cikti = np.dot(birlesik_cikti, W_o)
print(f"Multi-Head Attention Katmanının FİNAL ÇIKTISI (boyut: {final_cikti.shape}):\n{np.round(final_cikti, 2)}")

--- Adım 0: Ayarlar ve Hiperparametreler ---
Girdi Metni: 'akıllı köpek kırmızı topu kovaladı'

Model Boyutu (d_model): 6
Kafa Sayısı (num_heads): 6
Her Kafanın Boyutu (d_k): 6 / 6 = 1

Ortak Giriş Embedding Matrisi X (boyut: (5, 6)):
[[0.37 0.95 0.73 0.6  0.16 0.16]
 [0.06 0.87 0.6  0.71 0.02 0.97]
 [0.83 0.21 0.18 0.18 0.3  0.52]
 [0.43 0.29 0.61 0.14 0.29 0.37]
 [0.46 0.79 0.2  0.51 0.59 0.05]]


--- Adım 1: Her Kafa İçin AYRI Ağırlık Matrisleri Oluşturma ---
Farklılaşmanın başladığı yer tam olarak burasıdır!
6 kafa olduğu için, 6 set ayrı W_q, W_k, W_v matrisi oluşturulur.

--- BAŞLIK 1 AĞIRLIKLARI ---
W_q_1 (boyut: (6, 1)):
[[0.61]
 [0.17]
 [0.07]
 [0.95]
 [0.97]
 [0.81]]

W_k_1 (boyut: (6, 1)):
[[0.3 ]
 [0.1 ]
 [0.68]
 [0.44]
 [0.12]
 [0.5 ]]

--- BAŞLIK 2 AĞIRLIKLARI ---
W_q_2 (boyut: (6, 1)):
[[0.55]
 [0.18]
 [0.97]
 [0.78]
 [0.94]
 [0.89]]

W_k_2 (boyut: (6, 1)):
[[0.6 ]
 [0.92]
 [0.09]
 [0.2 ]
 [0.05]
 [0.33]]

--- BAŞLIK 3 AĞIRLIKLARI ---
W_q_3 (boyut: (6, 1)):
[[0.14]
 [0.8

## 5.1. MHA Örnek metin üzerinde çalışması

In [8]:
import numpy as np

# --- Adım 0: Ayarlar ve "Eğitilmiş" Model ---
print("--- Adım 0: Ayarlar ve 'Eğitilmiş' Multi-Head Model ---")
# Hiperparametreler
d_model = 8  # Modelin ana boyutu
num_heads = 2  # Kullanacağımız kafa sayısı
d_k = d_model // num_heads # Her kafanın kendi çalışma boyutu

print(f"Model Boyutu (d_model): {d_model}, Kafa Sayısı: {num_heads}, Her Kafa Boyutu (d_k): {d_k}\n")
np.random.seed(42) # Tekrarlanabilirlik için

# Her kafa için ayrı "eğitilmiş" ağırlık matrisleri
kafa_agirliklari = {}
for i in range(num_heads):
    kafa_agirliklari[f'kafa_{i+1}'] = {
        'W_q': np.round(np.random.rand(d_model, d_k), 2),
        'W_k': np.round(np.random.rand(d_model, d_k), 2),
        'W_v': np.round(np.random.rand(d_model, d_k), 2)
    }
print("Her kafa için ayrı W_q, W_k, W_v setleri yüklendi.\n")
print("="*60)


# --- Adım 1: Bilgi Bankasını (Context) İşleme ---
print("\n--- Adım 1: Bilgi Bankasının Her Kafa Tarafından Ayrı Ayrı İşlenmesi ---")
bilgi_bankasi_metni = "zeki siyah köpek tembel beyaz kediyi hızla kovaladı turkcell ile baglan hayata"
context_tokenler = bilgi_bankasi_metni.split()
print(f"Bilgi Bankası Metni: '{bilgi_bankasi_metni}'\n")

# Ortak girdi embedding matrisi
X_context = np.round(np.random.rand(len(context_tokenler), d_model), 2)

# Her kafanın kendi "hafızasını" (K ve V matrislerini) oluşturması
context_hafizasi = {}
for i in range(num_heads):
    kafa_adi = f'kafa_{i+1}'
    W_k = kafa_agirliklari[kafa_adi]['W_k']
    W_v = kafa_agirliklari[kafa_adi]['W_v']
    
    # Her kafa, ortak X_context'i kendi K ve V ağırlıklarıyla çarpar
    K_context = np.dot(X_context, W_k)
    V_context = np.dot(X_context, W_v)
    context_hafizasi[kafa_adi] = {'K': K_context, 'V': V_context}
    
    print(f"--- BAŞLIK {i+1}'in Hafızası ---")
    print(f"K_context_{i+1} (boyut: {K_context.shape}):\n{np.round(K_context, 2)}")
    print(f"V_context_{i+1} (boyut: {V_context.shape}):\n{np.round(V_context, 2)}\n")
print("Gördüğünüz gibi, her kafa bilgi bankasını kendi perspektifinden 'anladı' ve hafızasına kaydetti.")
print("="*60)


# --- Adım 2: Soruyu İşleme ve Cevap Bulma ---
print("\n--- Adım 2: Soru Geldi ve Her Kafa Cevap İçin Çalışıyor ---")
soru_metni = "turkcell ne yaptı"
sorgu_kelimesi = "turkcell"
print(f"Soru: '{soru_metni}'\nOdaklanılan Sorgu Kelimesi: '{sorgu_kelimesi}'\n")

# Soru için embedding
soru_token_index = 2 # "köpek" kelimesinin soru metninde olmadığını varsayalım, sadece anahtar kelime var.
# Normalde soru da embedding'den geçer, biz direkt X_context'ten alarak simüle edelim.
X_soru_kelimesi = X_context[soru_token_index] 

# Her kafa, soru kelimesi için kendi Q vektörünü üretir
soru_sorgulari = {}
for i in range(num_heads):
    kafa_adi = f'kafa_{i+1}'
    W_q = kafa_agirliklari[kafa_adi]['W_q']
    Q_vektoru = np.dot(X_soru_kelimesi, W_q)
    soru_sorgulari[kafa_adi] = Q_vektoru

# --- Adım 3: Her Kafanın Dikkatini Analiz Etme ---
print("\n--- Adım 3: Her Kafanın 'Düşünme Süreci' ve Cevabı ---")

final_cevaplar = {}
for i in range(num_heads):
    kafa_adi = f'kafa_{i+1}'
    print(f"\n########### BAŞLIK {i+1} Analiz Ediyor ###########\n")
    
    # 1. Kendi Soru Vektörünü (Q) ve Hafızasındaki Anahtar Vektörlerini (K) al
    Q_soru = soru_sorgulari[kafa_adi]
    K_context = context_hafizasi[kafa_adi]['K']
    print(f"1. BAŞLIK {i+1}, kendi ürettiği '{sorgu_kelimesi}' Q Vektörünü aldı: {np.round(Q_soru, 2)}")
    print("   ve bunu kendi hafızasındaki K matrisi ile karşılaştıracak.")
    
    # 2. Dikkat Skorlarını Hesapla
    dikkat_skorlari = np.dot(Q_soru, K_context.T)
    print("\n2. Dikkat Skorları (Sorgunun hafızadaki her kelimeyle ilgisi):")
    skor_raporu = "   " + "  ".join([f"'{token}':{skor:.2f}" for token, skor in zip(context_tokenler, dikkat_skorlari)])
    print(skor_raporu)

    # 3. Softmax ile Olasılıkları Bul
    dikkat_agirliklari = np.exp(dikkat_skorlari) / np.sum(np.exp(dikkat_skorlari))
    print("\n3. Softmax Ağırlıkları (Bu kafa en çok nereye odaklandı?):")
    agirlik_raporu = "   " + "  ".join([f"'{token}':{agirlik:.3f}" for token, agirlik in zip(context_tokenler, dikkat_agirliklari)])
    print(agirlik_raporu)
    
    # 4. Bu Kafanın Cevabını Bul
    cevap_index = np.argmax(dikkat_agirliklari)
    cevap_kelimesi = context_tokenler[cevap_index]
    final_cevaplar[kafa_adi] = cevap_kelimesi
    
    print(f"\n>>> BAŞLIK {i+1}'in Cevabı: '{cevap_kelimesi}' <<<")

print("\n\n================= FİNAL SONUÇ =================\n")
print(f"Soru: '{soru_metni}'")
print("\nModelin 'Uzmanlar Komitesi' (Her Kafa) şu sonuçlara ulaştı:")
for kafa, cevap in final_cevaplar.items():
    print(f"- {kafa.replace('_', ' ').capitalize()}: '{sorgu_kelimesi}' kelimesi en çok '{cevap}' kelimesiyle ilgili.")

print("\nÖZET: Farklı kafalar, farklı ağırlık setleri nedeniyle farklı özelliklere odaklandı.")
print("Bu örnekte, bir kafa eyleme ('kovaladı') odaklanırken, bir diğeri sıfata veya başka bir ilişkiye odaklanabilirdi.")
print("Gerçek bir LLM, bu farklı dikkat çıktılarından gelen zenginleştirilmiş 'Değer' (V) vektörlerini birleştirerek")
print(" 'Köpek, kediyi kovaladı.' gibi tam bir cümle üretir.")

--- Adım 0: Ayarlar ve 'Eğitilmiş' Multi-Head Model ---
Model Boyutu (d_model): 8, Kafa Sayısı: 2, Her Kafa Boyutu (d_k): 4

Her kafa için ayrı W_q, W_k, W_v setleri yüklendi.


--- Adım 1: Bilgi Bankasının Her Kafa Tarafından Ayrı Ayrı İşlenmesi ---
Bilgi Bankası Metni: 'zeki siyah köpek tembel beyaz kediyi hızla kovaladı turkcell ile baglan hayata'

--- BAŞLIK 1'in Hafızası ---
K_context_1 (boyut: (12, 4)):
[[2.02 2.64 3.36 3.47]
 [0.93 1.61 2.11 1.92]
 [1.02 1.28 1.35 1.97]
 [1.6  2.02 2.41 2.83]
 [1.66 2.04 2.27 3.12]
 [0.7  1.42 1.59 1.76]
 [1.86 2.87 3.1  3.56]
 [1.46 2.02 3.13 2.83]
 [1.86 2.48 3.07 3.27]
 [1.41 2.43 2.83 3.15]
 [1.95 2.58 3.5  3.57]
 [0.84 1.51 2.19 2.1 ]]
V_context_1 (boyut: (12, 4)):
[[2.86 3.02 2.58 2.77]
 [2.   1.42 1.32 1.19]
 [1.99 1.61 1.41 1.33]
 [1.91 2.36 2.32 1.64]
 [2.45 2.62 2.29 2.28]
 [1.72 1.6  1.18 1.37]
 [2.05 3.28 3.13 3.03]
 [2.04 2.65 2.1  1.85]
 [1.75 2.74 2.46 2.35]
 [2.58 2.98 2.49 2.63]
 [2.82 3.31 2.8  2.66]
 [1.69 2.03 1.38 1.73]]

--

## 5.2. MHA Özet Örnek Kod

In [5]:
import torch
import torch.nn.functional as F
import numpy as np

def single_head_attention(Q, K, V):
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(d_k)
    weights = F.softmax(scores, dim=-1)
    output = torch.matmul(weights, V)
    return output, weights

def multi_head_attention(Q, K, V, num_heads):
    batch_size, seq_len, d_model = Q.size()
    assert d_model % num_heads == 0, "d_model num_heads'e tam bölünmelidir"
    d_k = d_model // num_heads

    # Lineer projeksiyonlar (başlık sayısı kadar)
    Q = Q.view(batch_size, seq_len, num_heads, d_k).transpose(1, 2)  # (B, h, S, d_k)
    K = K.view(batch_size, seq_len, num_heads, d_k).transpose(1, 2)
    V = V.view(batch_size, seq_len, num_heads, d_k).transpose(1, 2)

    scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(d_k)
    weights = F.softmax(scores, dim=-1)
    print("Her head için attention score matrisleri:")
    for i, head_weights in enumerate(weights[0]):
        print(f"\nHead {i+1}:")
        print(np.round(head_weights.detach().numpy(), 2))

    attended = torch.matmul(weights, V)  # (B, h, S, d_k)
    concat = attended.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)
    return concat, weights

# Simülasyon: 1 örnek, 4 kelime, 8 boyutlu embedding
torch.manual_seed(42)
x = torch.randn(1, 4, 8)  # (batch, seq_len, d_model)

# Tek başlı attention
Q = K = V = x.clone()
output_single, weights_single = single_head_attention(Q[0], K[0], V[0])
print("\n--- Tek Başlı Attention ---")
print("Attention çıktısı:", np.round(output_single.detach().numpy(), 2))

# Çok başlı attention (2 head)
print("\n--- Multi-Head Attention (2 Head) ---")
output_multi, weights_multi = multi_head_attention(Q, K, V, num_heads=2)
print("MHA Çıktısı:", np.round(output_multi[0].detach().numpy(), 2))



--- Tek Başlı Attention ---
Attention çıktısı: [[ 1.9   1.49  0.89 -2.08  0.66 -1.22 -0.05 -1.57]
 [-0.18  1.54 -0.16 -1.28 -0.5  -0.56 -0.58  0.47]
 [ 1.53  0.13 -0.31  0.49 -0.66  0.84  0.59  1.48]
 [ 1.28  0.99  0.3   0.66 -0.32  0.15 -0.02  0.85]]

--- Multi-Head Attention (2 Head) ---
Her head için attention score matrisleri:

Head 1:
[[0.96 0.02 0.01 0.01]
 [0.28 0.68 0.02 0.03]
 [0.21 0.04 0.47 0.29]
 [0.13 0.03 0.14 0.7 ]]

Head 2:
[[0.89 0.06 0.01 0.04]
 [0.1  0.43 0.21 0.26]
 [0.01 0.08 0.79 0.12]
 [0.08 0.29 0.36 0.27]]
MHA Çıktısı: [[ 1.86  1.48  0.86 -2.03  0.55 -1.12 -0.08 -1.34]
 [ 0.09  1.56 -0.01 -1.49 -0.47 -0.12 -0.23  0.76]
 [ 1.51  0.67  0.12  0.09 -0.68  0.81  0.54  1.49]
 [ 1.36  1.13  0.47  0.68 -0.49  0.14 -0.    0.93]]


In [6]:
import torch
import torch.nn.functional as F
import numpy as np

# Tek başlı attention
def single_head_attention(Q, K, V):
    d_k = Q.shape[-1]
    scores = torch.matmul(Q, K.T) / np.sqrt(d_k)
    weights = F.softmax(scores, dim=-1)
    output = torch.matmul(weights, V)
    return output, weights

# Çok başlı attention
def multi_head_attention(Q, K, V, num_heads=2):
    d_model = Q.shape[-1]
    assert d_model % num_heads == 0
    d_k = d_model // num_heads

    # Split and stack Q, K, V
    Q_heads = Q.reshape(Q.shape[0], num_heads, d_k)
    K_heads = K.reshape(K.shape[0], num_heads, d_k)
    V_heads = V.reshape(V.shape[0], num_heads, d_k)

    weights_all = []
    head_outputs = []
    for h in range(num_heads):
        Qh, Kh, Vh = Q_heads[:, h], K_heads[:, h], V_heads[:, h]
        scores = torch.matmul(Qh, Kh.T) / np.sqrt(d_k)
        weights = F.softmax(scores, dim=-1)
        output = torch.matmul(weights, Vh)
        weights_all.append(weights.unsqueeze(1))
        head_outputs.append(output)
    
    # Concatenate head outputs
    final_output = torch.cat(head_outputs, dim=-1)
    all_weights = torch.cat(weights_all, dim=1)
    return final_output, all_weights

# Soru ve bağlam
tokens = ["türkiye", "ankara", "başkent", "neresidir"]
context = ["türkiye", "başkent", "ankara"]
question = ["başkent", "neresidir"]

# Sözlük ve embedding
token_to_idx = {tok: i for i, tok in enumerate(tokens)}
torch.manual_seed(0)
embedding = torch.randn(len(tokens), 4)  # 4 boyutlu vektör, 2 head için

context_embed = embedding[[token_to_idx[t] for t in context]]
question_embed = embedding[[token_to_idx[t] for t in question]]

print("\n Soru:", question)
print("Bağlam:", context)

# Tek başlı attention
Q = question_embed
K = V = context_embed
single_out, single_weights = single_head_attention(Q, K, V)
print("\n🔹 Tek Başlı Attention Ağırlıkları:")
print(np.round(single_weights.detach().numpy(), 2))

# Çok başlı attention
multi_out, multi_weights = multi_head_attention(Q, K, V, num_heads=2)
print("\n Çok Başlı Attention Head 1:")
print(np.round(multi_weights[:, 0, :].detach().numpy(), 2))
print("\n Çok Başlı Attention Head 2:")
print(np.round(multi_weights[:, 1, :].detach().numpy(), 2))



 Soru: ['başkent', 'neresidir']
Bağlam: ['türkiye', 'başkent', 'ankara']

🔹 Tek Başlı Attention Ağırlıkları:
[[0.33 0.56 0.11]
 [0.15 0.2  0.65]]

 Çok Başlı Attention Head 1:
[[0.35 0.54 0.11]
 [0.13 0.13 0.75]]

 Çok Başlı Attention Head 2:
[[0.33 0.45 0.22]
 [0.27 0.38 0.35]]
